# 05 — Baselines ML & Forecast Diagnostics (V9 forecast traffic-memory)

**Thèse :** Explainable LLM Framework for Tourist-Traffic Forecasting in Phuket

**Branch:** `v9_forecast_traffic_memory_scratch`

This notebook is especially important for V9 because it tells us **what actually predicts the target best** before we decide how to train the LLM.

**V9 purpose:**
1. Recompute baselines after the upstream dataset changes from `03_prepare_dataset.ipynb`
2. Re-check whether short traffic memory helps classical forecasting
3. Use SHAP / XGBoost signals to justify the V9 prompt design
4. Keep a strong numeric benchmark next to the LLM branch

**Baselines built here:**
- **LV** — Last Value
- **HA** — Historical Average
- **SN** — Seasonal Naive
- **XGB** — XGBoost with ML features
- **SHAP** — feature importance for XGBoost

**Reference target for tuning / SHAP / plots:** `tt_ratio_Weekday_AM1`

> **V9 note:** this notebook is the main reason we pushed the LLM branch toward traffic-memory-first prompting. The strongest predictive signals consistently come from traffic-internal variables, not from a purely exogenous narrative context.


In [1]:
# ── 0. Install missing packages ────────────────────────────────────────────────
import subprocess, sys

REQUIRED = [
    ('xgboost',      'xgboost'),
    ('shap',         'shap'),
    ('sklearn',      'scikit-learn'),
    ('matplotlib',   'matplotlib'),
    ('optuna',       'optuna'),
]

for import_name, pkg_name in REQUIRED:
    try:
        __import__(import_name)
        print(f'  ✓ {pkg_name}')
    except ImportError:
        print(f'  Installing {pkg_name}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg_name], check=True, capture_output=True)
        print(f'  ✓ {pkg_name} installed')

  ✓ xgboost


/Users/mathieuzilli/Desktop/Thai/TFE_Phuket/tomtom_phuket/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ✓ shap
  ✓ scikit-learn
  ✓ matplotlib
  ✓ optuna


In [2]:
# ── 1. Imports ─────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # no display needed — save to file
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder

import xgboost as xgb
import shap
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import os, json, pathlib

DATA_DIR = pathlib.Path('../data')
FIG_DIR   = pathlib.Path('../final_data/figures')
TABLE_DIR = pathlib.Path('../final_data/tables')

print('All imports OK')
print(f'XGBoost version: {xgb.__version__}')
print(f'SHAP version: {shap.__version__}')

All imports OK
XGBoost version: 3.2.0
SHAP version: 0.51.0


In [3]:
# ── 2. Load data ──────────────────────────────────────────────────────────────
train  = pd.read_csv(DATA_DIR / 'ml_train.csv')
val    = pd.read_csv(DATA_DIR / 'ml_val.csv')
test   = pd.read_csv(DATA_DIR / 'ml_test.csv')
master = pd.read_csv(DATA_DIR / 'phuket_master.csv')   # for Seasonal Naive lookup

print(f'Train shape  : {train.shape}')
print(f'Val   shape  : {val.shape}')
print(f'Test  shape  : {test.shape}')
print(f'Master shape : {master.shape}')

print(f'\nTrain columns ({len(train.columns)}):')
for i, c in enumerate(train.columns):
    print(f'  [{i:02d}] {c}')

print(f'\nCorridors in train: {sorted(train["corr_id"].unique())}')
print(f'Train date range:')
for cid, grp in train.groupby('corr_id'):
    s = grp.sort_values(['year','month'])
    print(f'  {cid}: {int(s.iloc[0]["year"])}-{int(s.iloc[0]["month"]):02d} → {int(s.iloc[-1]["year"])}-{int(s.iloc[-1]["month"]):02d}  ({len(grp)} rows)')

Train shape  : (36, 106)
Val   shape  : (24, 106)
Test  shape  : (24, 106)
Master shape : (96, 101)

Train columns (106):
  [00] spd_Weekday_AM1
  [01] spd_Weekday_AM2
  [02] spd_Weekday_Afternoon
  [03] spd_Weekday_Early
  [04] spd_Weekday_Evening
  [05] spd_Weekday_LateNight
  [06] spd_Weekday_Midday
  [07] spd_Weekday_Morning
  [08] spd_Weekday_Night
  [09] spd_Weekday_PM1
  [10] spd_Weekday_PM2
  [11] spd_Weekday_PrePM
  [12] spd_Weekend_AM1
  [13] spd_Weekend_AM2
  [14] spd_Weekend_Afternoon
  [15] spd_Weekend_Early
  [16] spd_Weekend_Evening
  [17] spd_Weekend_LateNight
  [18] spd_Weekend_Midday
  [19] spd_Weekend_Morning
  [20] spd_Weekend_Night
  [21] spd_Weekend_PM1
  [22] spd_Weekend_PM2
  [23] spd_Weekend_PrePM
  [24] tt_ratio_Weekday_AM1
  [25] tt_ratio_Weekday_AM2
  [26] tt_ratio_Weekday_Afternoon
  [27] tt_ratio_Weekday_Early
  [28] tt_ratio_Weekday_Evening
  [29] tt_ratio_Weekday_LateNight
  [30] tt_ratio_Weekday_Midday
  [31] tt_ratio_Weekday_Morning
  [32] tt_ratio_Wee

In [4]:
# ── 3. Target definitions ─────────────────────────────────────────────────────
TARGETS = [
    'tt_ratio_Weekday_AM1', 'tt_ratio_Weekday_PM1',
    'tt_ratio_Weekday_Midday', 'tt_ratio_Weekend_AM1',
    'pti_mean', 'pti_max'
]
PRIMARY = 'tt_ratio_Weekday_AM1'

# Verify all targets present
missing_t = [t for t in TARGETS if t not in train.columns]
if missing_t:
    print(f'⚠️  Missing targets: {missing_t}')
else:
    print('✅  All targets present')

print(f'\n=== Target statistics per corridor (TRAIN) ===')
for t in TARGETS:
    stats = train.groupby('corr_id')[t].agg(['mean', 'std', 'min', 'max']).round(4)
    print(f'\n  {t}:')
    print(stats.to_string())


✅  All targets present

=== Target statistics per corridor (TRAIN) ===

  tt_ratio_Weekday_AM1:
           mean     std   min   max
corr_id                            
0        1.2344  0.0816  1.11  1.32
1        1.1778  0.0931  0.99  1.32
2        1.5956  0.1631  1.33  1.79
3        1.5544  0.2244  1.13  1.78

  tt_ratio_Weekday_PM1:
           mean     std   min   max
corr_id                            
0        1.5078  0.0871  1.33  1.60
1        1.3311  0.0979  1.19  1.48
2        1.8367  0.1576  1.58  2.00
3        1.5422  0.1092  1.31  1.69

  tt_ratio_Weekday_Midday:
           mean     std   min   max
corr_id                            
0        1.3033  0.0773  1.19  1.42
1        1.2456  0.0559  1.17  1.32
2        1.5933  0.0723  1.50  1.73
3        1.3611  0.0870  1.29  1.54

  tt_ratio_Weekend_AM1:
           mean     std   min   max
corr_id                            
0        1.1511  0.0936  1.01  1.29
1        0.9789  0.0408  0.91  1.03
2        1.2922  0.0574  1.20  1.3

In [5]:
# ── 4. Metrics helper ─────────────────────────────────────────────────────────
def mape_safe(y_true, y_pred):
    """MAPE avoiding division by zero."""
    mask = np.abs(y_true) > 1e-10
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

def compute_metrics(y_true, y_pred, label=''):
    """Compute MAE, RMSE, MAPE and print."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    # Drop NaN predictions
    valid = ~(np.isnan(y_true) | np.isnan(y_pred))
    n_valid = valid.sum()
    if n_valid == 0:
        print(f'{label:55s}  n=0  (no valid predictions)')
        return {'label': label, 'MAE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan, 'n': 0}
    yt, yp = y_true[valid], y_pred[valid]
    mae  = float(mean_absolute_error(yt, yp))
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    mape = mape_safe(yt, yp)
    print(f'{label:55s}  n={n_valid:3d}  MAE={mae:.6f}  RMSE={rmse:.6f}  MAPE={mape:.4f}%')
    return {'label': label, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'n': n_valid}

print('Metrics helper defined OK')

Metrics helper defined OK


In [6]:
# ── 5. Baseline 1 — Last Value (LV) ──────────────────────────────────────────
# For each corridor: LV = the last observed value in train
# On val: last train month per corridor
# On test: same (simulating real deployment — only train is known before val/test)

lv_lookup = (
    train.sort_values(['corr_id','year','month'])
         .groupby('corr_id')[TARGETS]
         .last()
         .reset_index()
         .rename(columns={t: f'{t}_lv' for t in TARGETS})
)

print('LV lookup (last train value per corridor):')
print(lv_lookup.to_string())

val_lv  = val.merge(lv_lookup,  on='corr_id', how='left')
test_lv = test.merge(lv_lookup, on='corr_id', how='left')

results_lv = []
print('\n=== LV Baseline — Val Set ===')
for t in TARGETS:
    m = compute_metrics(val_lv[t].values, val_lv[f'{t}_lv'].values, f'LV|val|{t}')
    results_lv.append({**m, 'baseline': 'LV', 'split': 'val', 'target': t})

print('\n=== LV Baseline — Test Set ===')
for t in TARGETS:
    m = compute_metrics(test_lv[t].values, test_lv[f'{t}_lv'].values, f'LV|test|{t}')
    results_lv.append({**m, 'baseline': 'LV', 'split': 'test', 'target': t})

LV lookup (last train value per corridor):
   corr_id  tt_ratio_Weekday_AM1_lv  tt_ratio_Weekday_PM1_lv  tt_ratio_Weekday_Midday_lv  tt_ratio_Weekend_AM1_lv  pti_mean_lv  pti_max_lv
0        0                     1.28                     1.53                        1.42                     1.17        2.614        3.40
1        1                     1.18                     1.41                        1.31                     0.96        2.794        3.92
2        2                     1.79                     2.00                        1.73                     1.39        3.952        5.53
3        3                     1.54                     1.58                        1.31                     1.19        2.441        3.48

=== LV Baseline — Val Set ===
LV|val|tt_ratio_Weekday_AM1                              n= 24  MAE=0.109167  RMSE=0.155322  MAPE=7.9286%
LV|val|tt_ratio_Weekday_PM1                              n= 24  MAE=0.088750  RMSE=0.106439  MAPE=5.6328%
LV|val|tt_ratio_Wee

In [7]:
# ── 6. Baseline 2 — Historical Average (HA) ───────────────────────────────────
# HA: per (corridor, month), average of that month across all training years
# NOTE: train = 2023 only (1 year) → HA lookup has 1 value per (corridor, month)
#       → HA ≡ SN on val/test months that exist in 2023 training data

ha_lookup = (
    train.groupby(['corr_id','month'])[TARGETS]
         .mean()
         .reset_index()
         .rename(columns={t: f'{t}_ha' for t in TARGETS})
)

print('HA lookup shape:', ha_lookup.shape)
print('HA lookup sample (corr_id + months available):')
print(ha_lookup[['corr_id','month']].groupby('corr_id')['month'].apply(list).to_string())

val_ha  = val.merge(ha_lookup,  on=['corr_id','month'], how='left')
test_ha = test.merge(ha_lookup, on=['corr_id','month'], how='left')

# Check coverage
# NOTE: HA val will have NaN for Jan/Feb/Mar 2024 — these months absent from train (Apr-Dec 2023)
# → HA val metrics computed on n=12 only (Apr-Jun 2024), not n=24
for split_name, df_ha in [('val', val_ha), ('test', test_ha)]:
    na_rows = df_ha[f'{TARGETS[0]}_ha'].isna().sum()
    print(f'  HA {split_name}: {na_rows}/{len(df_ha)} rows with no HA prediction (months not in train)')

results_ha = []
print('\n=== HA Baseline — Val Set ===')
for t in TARGETS:
    m = compute_metrics(val_ha[t].values, val_ha[f'{t}_ha'].values, f'HA|val|{t}')
    results_ha.append({**m, 'baseline': 'HA', 'split': 'val', 'target': t})

print('\n=== HA Baseline — Test Set ===')
for t in TARGETS:
    m = compute_metrics(test_ha[t].values, test_ha[f'{t}_ha'].values, f'HA|test|{t}')
    results_ha.append({**m, 'baseline': 'HA', 'split': 'test', 'target': t})

HA lookup shape: (36, 8)
HA lookup sample (corr_id + months available):
corr_id
0    [4, 5, 6, 7, 8, 9, 10, 11, 12]
1    [4, 5, 6, 7, 8, 9, 10, 11, 12]
2    [4, 5, 6, 7, 8, 9, 10, 11, 12]
3    [4, 5, 6, 7, 8, 9, 10, 11, 12]
  HA val: 12/24 rows with no HA prediction (months not in train)
  HA test: 0/24 rows with no HA prediction (months not in train)

=== HA Baseline — Val Set ===
HA|val|tt_ratio_Weekday_AM1                              n= 12  MAE=0.090000  RMSE=0.106927  MAPE=6.3831%
HA|val|tt_ratio_Weekday_PM1                              n= 12  MAE=0.083333  RMSE=0.095044  MAPE=5.4614%
HA|val|tt_ratio_Weekday_Midday                           n= 12  MAE=0.100833  RMSE=0.118849  MAPE=7.1709%
HA|val|tt_ratio_Weekend_AM1                              n= 12  MAE=0.055833  RMSE=0.084705  MAPE=4.6411%
HA|val|pti_mean                                          n= 12  MAE=0.244333  RMSE=0.314881  MAPE=7.8504%
HA|val|pti_max                                           n= 12  MAE=0.285833  RMSE=0.

In [8]:
# ── 7. Baseline 3 — Seasonal Naive (SN) ──────────────────────────────────────
# SN: for (corridor, year=Y, month=M), predict value from (corridor, year=Y-1, month=M)
# Uses master.csv as lookup source (covers 2023-2024)

sn_src = master[['corr_id','year','month'] + TARGETS].copy()
sn_src = sn_src.rename(columns={t: f'{t}_sn' for t in TARGETS})
# Shift year by +1 so (year=2023, month=M) maps to target (year=2024, month=M)
sn_src['year'] = sn_src['year'] + 1

print('SN source shape:', sn_src.shape)
print('SN year range after shift:', sn_src['year'].min(), '–', sn_src['year'].max())

val_sn  = val.merge(sn_src,  on=['corr_id','year','month'], how='left')
test_sn = test.merge(sn_src, on=['corr_id','year','month'], how='left')

for split_name, df_sn in [('val', val_sn), ('test', test_sn)]:
    na_rows = df_sn[f'{TARGETS[0]}_sn'].isna().sum()
    print(f'  SN {split_name}: {na_rows}/{len(df_sn)} rows with no SN prediction (year-1 not in master)')

results_sn = []
print('\n=== Seasonal Naive — Val Set ===')
for t in TARGETS:
    m = compute_metrics(val_sn[t].values, val_sn[f'{t}_sn'].values, f'SN|val|{t}')
    results_sn.append({**m, 'baseline': 'SN', 'split': 'val', 'target': t})

print('\n=== Seasonal Naive — Test Set ===')
for t in TARGETS:
    m = compute_metrics(test_sn[t].values, test_sn[f'{t}_sn'].values, f'SN|test|{t}')
    results_sn.append({**m, 'baseline': 'SN', 'split': 'test', 'target': t})

SN source shape: (96, 9)
SN year range after shift: 2024 – 2025
  SN val: 0/24 rows with no SN prediction (year-1 not in master)
  SN test: 0/24 rows with no SN prediction (year-1 not in master)

=== Seasonal Naive — Val Set ===
SN|val|tt_ratio_Weekday_AM1                              n= 24  MAE=0.093750  RMSE=0.116315  MAPE=6.6150%
SN|val|tt_ratio_Weekday_PM1                              n= 24  MAE=0.100417  RMSE=0.121054  MAPE=6.3489%
SN|val|tt_ratio_Weekday_Midday                           n= 24  MAE=0.091667  RMSE=0.118533  MAPE=6.3920%
SN|val|tt_ratio_Weekend_AM1                              n= 24  MAE=0.067500  RMSE=0.104043  MAPE=5.5090%
SN|val|pti_mean                                          n= 24  MAE=0.246792  RMSE=0.324959  MAPE=7.8238%
SN|val|pti_max                                           n= 24  MAE=0.301250  RMSE=0.375183  MAPE=7.2459%

=== Seasonal Naive — Test Set ===
SN|test|tt_ratio_Weekday_AM1                             n= 24  MAE=0.084583  RMSE=0.112565  MAPE=5.

In [9]:
# ── 8. XGBoost — Feature set definition ───────────────────────────────────────

# Columns to exclude from features
EXCLUDE_FROM_FEATURES = set(
    TARGETS
    + ['split', 'corridor', 'season']  # metadata / string columns
    + ['flt_intl_arrivals', 'flt_intl_departures', 'flt_domestic', 'wx_rain_days',
       'soc_phuket_beach', 'soc_phuket_hotel', 'soc_phuket_vacation', 'soc_phuket_flight',
    ]  # ^ redundant/leaky features removed (multicollinear exogenous)
)

FEATURE_COLS = [c for c in train.columns if c not in EXCLUDE_FROM_FEATURES]

# Verify all features are numeric (after encoding corr_id)
non_numeric = [c for c in FEATURE_COLS if c != 'corr_id' and train[c].dtype == object]
print(f'Non-numeric feature columns (before encoding): {non_numeric}')
if non_numeric:
    print('  → These will be dropped to avoid XGBoost errors')
    FEATURE_COLS = [c for c in FEATURE_COLS if c not in non_numeric]

print(f'Feature count: {len(FEATURE_COLS)}')
print('Feature columns:')
for i, c in enumerate(FEATURE_COLS):
    print(f'  [{i:02d}] {c}  dtype={train[c].dtype}')

# Encode corr_id (string → integer)
le = LabelEncoder().fit(train['corr_id'])
print(f'\ncorr_id encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}')

def prepare_X(df):
    X = df[FEATURE_COLS].copy()
    X['corr_id'] = le.transform(X['corr_id'])
    return X.astype(float)  # ensure all numeric

X_train = prepare_X(train)
X_val   = prepare_X(val)
X_test  = prepare_X(test)

print(f'\nX_train: {X_train.shape}  |  X_val: {X_val.shape}  |  X_test: {X_test.shape}')
print(f'NaN in X_train: {X_train.isna().sum().sum()}')
print(f'NaN in X_val:   {X_val.isna().sum().sum()}')
print(f'NaN in X_test:  {X_test.isna().sum().sum()}')
print(f'Dtypes check: {X_train.dtypes.unique().tolist()}')

Non-numeric feature columns (before encoding): []
Feature count: 97
Feature columns:
  [00] spd_Weekday_AM1  dtype=float64
  [01] spd_Weekday_AM2  dtype=float64
  [02] spd_Weekday_Afternoon  dtype=float64
  [03] spd_Weekday_Early  dtype=float64
  [04] spd_Weekday_Evening  dtype=float64
  [05] spd_Weekday_LateNight  dtype=float64
  [06] spd_Weekday_Midday  dtype=float64
  [07] spd_Weekday_Morning  dtype=float64
  [08] spd_Weekday_Night  dtype=float64
  [09] spd_Weekday_PM1  dtype=float64
  [10] spd_Weekday_PM2  dtype=float64
  [11] spd_Weekday_PrePM  dtype=float64
  [12] spd_Weekend_AM1  dtype=float64
  [13] spd_Weekend_AM2  dtype=float64
  [14] spd_Weekend_Afternoon  dtype=float64
  [15] spd_Weekend_Early  dtype=float64
  [16] spd_Weekend_Evening  dtype=float64
  [17] spd_Weekend_LateNight  dtype=float64
  [18] spd_Weekend_Midday  dtype=float64
  [19] spd_Weekend_Morning  dtype=float64
  [20] spd_Weekend_Night  dtype=float64
  [21] spd_Weekend_PM1  dtype=float64
  [22] spd_Weekend_PM2 

In [10]:
# ── 9. XGBoost — Optuna hyperparameter search (primary target) ─────────────────

y_train = train[PRIMARY].values
y_val   = val[PRIMARY].values

def xgb_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 50, 300),
        'max_depth':        trial.suggest_int('max_depth', 2, 6),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'verbosity': 0, 'random_state': 42
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return mean_absolute_error(y_val, model.predict(X_val))

study = optuna.create_study(direction='minimize')
study.optimize(xgb_objective, n_trials=50, show_progress_bar=False)

print(f'Optuna best trial MAE (val): {study.best_value:.6f}')
print(f'Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

Optuna best trial MAE (val): 0.039173
Best params:
  n_estimators: 145
  max_depth: 5
  learning_rate: 0.11080260806581783
  subsample: 0.6436141080972069
  colsample_bytree: 0.7982467096179725
  min_child_weight: 6
  reg_alpha: 0.127797373495787
  reg_lambda: 0.0029144129121699744


In [11]:
# ── 10. XGBoost — Train final model with best params ─────────────────────────

best_params = {**study.best_params, 'verbosity': 0, 'random_state': 42}
model_xgb = xgb.XGBRegressor(**best_params)
model_xgb.fit(X_train, y_train)

print(f'=== XGBoost ({PRIMARY}) ===')
for split_name, X_s, y_s in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, test[PRIMARY].values)]:
    preds = model_xgb.predict(X_s)
    compute_metrics(y_s, preds, f'XGB|{split_name}|{PRIMARY}')

# Train models for ALL targets
xgb_models = {PRIMARY: model_xgb}
results_xgb = []

print('\n=== XGBoost — All targets (default params) ===')
for t in TARGETS:
    if t == PRIMARY:
        m_val  = compute_metrics(y_val, model_xgb.predict(X_val), f'XGB|val|{t}')
        m_test = compute_metrics(test[t].values, model_xgb.predict(X_test), f'XGB|test|{t}')
        results_xgb.append({**m_val,  'baseline': 'XGB', 'split': 'val',  'target': t})
        results_xgb.append({**m_test, 'baseline': 'XGB', 'split': 'test', 'target': t})
    else:
        # Use best_params from primary target (good enough for all targets)
        m = xgb.XGBRegressor(**best_params)
        m.fit(X_train, train[t].values)
        xgb_models[t] = m
        m_val  = compute_metrics(val[t].values,  m.predict(X_val),  f'XGB|val|{t}')
        m_test = compute_metrics(test[t].values, m.predict(X_test), f'XGB|test|{t}')
        results_xgb.append({**m_val,  'baseline': 'XGB', 'split': 'val',  'target': t})
        results_xgb.append({**m_test, 'baseline': 'XGB', 'split': 'test', 'target': t})

=== XGBoost (tt_ratio_Weekday_AM1) ===
XGB|train|tt_ratio_Weekday_AM1                           n= 36  MAE=0.012928  RMSE=0.023531  MAPE=0.9583%
XGB|val|tt_ratio_Weekday_AM1                             n= 24  MAE=0.039173  RMSE=0.057412  MAPE=2.7300%
XGB|test|tt_ratio_Weekday_AM1                            n= 24  MAE=0.046312  RMSE=0.075358  MAPE=3.1378%

=== XGBoost — All targets (default params) ===
XGB|val|tt_ratio_Weekday_AM1                             n= 24  MAE=0.039173  RMSE=0.057412  MAPE=2.7300%
XGB|test|tt_ratio_Weekday_AM1                            n= 24  MAE=0.046312  RMSE=0.075358  MAPE=3.1378%


XGB|val|tt_ratio_Weekday_PM1                             n= 24  MAE=0.047422  RMSE=0.057377  MAPE=2.9762%
XGB|test|tt_ratio_Weekday_PM1                            n= 24  MAE=0.060745  RMSE=0.076448  MAPE=3.7728%


XGB|val|tt_ratio_Weekday_Midday                          n= 24  MAE=0.033593  RMSE=0.046130  MAPE=2.2690%
XGB|test|tt_ratio_Weekday_Midday                         n= 24  MAE=0.068526  RMSE=0.089238  MAPE=4.4948%


XGB|val|tt_ratio_Weekend_AM1                             n= 24  MAE=0.034339  RMSE=0.058358  MAPE=2.8322%
XGB|test|tt_ratio_Weekend_AM1                            n= 24  MAE=0.041688  RMSE=0.060858  MAPE=3.2080%


XGB|val|pti_mean                                         n= 24  MAE=0.081092  RMSE=0.112107  MAPE=2.6028%
XGB|test|pti_mean                                        n= 24  MAE=0.097632  RMSE=0.141760  MAPE=2.9165%


XGB|val|pti_max                                          n= 24  MAE=0.131708  RMSE=0.175075  MAPE=3.2715%
XGB|test|pti_max                                         n= 24  MAE=0.172892  RMSE=0.234978  MAPE=4.2865%


In [12]:
# ── 11. SHAP analysis (primary target, XGBoost) ───────────────────────────────

# Use val set for SHAP (24 rows — manageable)
explainer    = shap.TreeExplainer(model_xgb)
shap_vals    = explainer.shap_values(X_val)
shap_abs_mean = pd.Series(np.abs(shap_vals).mean(axis=0), index=FEATURE_COLS)

print(f'=== SHAP Mean |value| — Top 20 features for {PRIMARY} (val set) ===')
top20 = shap_abs_mean.sort_values(ascending=False).head(20)
for feat, imp in top20.items():
    print(f'  {feat:40s}  {imp:.6f}')

print(f'\nBottom 10 features (lowest SHAP importance):')
bot10 = shap_abs_mean.sort_values(ascending=True).head(10)
for feat, imp in bot10.items():
    print(f'  {feat:40s}  {imp:.6f}')

# Save SHAP summary bar plot
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_vals, X_val, plot_type='bar', show=False, max_display=20)
plt.title(f'SHAP Feature Importance — {PRIMARY} (XGBoost, val set)', fontsize=12)
plt.tight_layout()
fig_path = FIG_DIR / 'shap_importance_bar.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'\nSHAP bar plot saved: {fig_path}')

# Save SHAP beeswarm plot
fig2, ax2 = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_vals, X_val, show=False, max_display=20)
plt.title(f'SHAP Beeswarm — {PRIMARY} (XGBoost, val set)', fontsize=12)
plt.tight_layout()
fig2_path = FIG_DIR / 'shap_beeswarm.png'
plt.savefig(fig2_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'SHAP beeswarm plot saved: {fig2_path}')

=== SHAP Mean |value| — Top 20 features for tt_ratio_Weekday_AM1 (val set) ===
  pti_Weekday_AM1                           0.148170
  tt_ratio_Weekday_PrePM                    0.027127
  spd_Weekday_AM1                           0.023955
  tt_ratio_Weekday_AM2                      0.015368
  pti_Weekday_AM2                           0.014095
  pti_Weekday_PrePM                         0.003876
  tt_ratio_Weekend_Night                    0.003863
  tt_ratio_Weekday_Early                    0.003567
  soc_phuket_l2                             0.003107
  pti_Weekend_Evening                       0.002015
  spd_Weekday_Early                         0.001988
  tt_ratio_Weekend_AM2                      0.001979
  spd_Weekday_AM2                           0.001614
  pti_Weekday_PM2                           0.001551
  tt_ratio_Weekend_Midday                   0.001417
  pti_Weekend_AM2                           0.001267
  tt_ratio_Weekday_Afternoon                0.000994
  pti_Weekday_Evenin


SHAP bar plot saved: ../final_data/figures/shap_importance_bar.png


SHAP beeswarm plot saved: ../final_data/figures/shap_beeswarm.png


In [13]:
# ── 12. SHAP — Per-corridor waterfall (one sample each) ───────────────────────

corridors_in_val = sorted(val['corr_id'].unique())
print(f'Corridors in val: {corridors_in_val}')

for cid in corridors_in_val:
    idx_list = val.index[val['corr_id'] == cid].tolist()
    if not idx_list:
        continue
    # Pick the first sample for this corridor in val
    local_idx = val.index.get_loc(idx_list[0])
    
    sv = shap.Explanation(
        values       = shap_vals[local_idx],
        base_values  = explainer.expected_value,
        data         = X_val.iloc[local_idx].values,
        feature_names= FEATURE_COLS
    )
    
    fig3 = plt.figure(figsize=(12, 5))
    shap.plots.waterfall(sv, max_display=12, show=False)
    plt.title(f'SHAP Waterfall — {cid} (first val sample)', fontsize=11)
    plt.tight_layout()
    wf_path = FIG_DIR / f'shap_waterfall_{cid}.png'
    plt.savefig(wf_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Waterfall saved: {wf_path}')
    
    # Print top features for this corridor
    top5 = pd.Series(shap_vals[local_idx], index=FEATURE_COLS).abs().sort_values(ascending=False).head(5)
    print(f'  Top 5 SHAP features for {cid}:')
    for feat, imp in top5.items():
        print(f'    {feat:40s}  {imp:.6f}')

Corridors in val: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


Waterfall saved: ../final_data/figures/shap_waterfall_0.png
  Top 5 SHAP features for 0:
    pti_Weekday_AM1                           0.128824
    pti_Weekday_AM2                           0.022012
    spd_Weekday_AM1                           0.020492
    tt_ratio_Weekday_AM2                      0.013854
    tt_ratio_Weekday_PrePM                    0.013536


Waterfall saved: ../final_data/figures/shap_waterfall_1.png
  Top 5 SHAP features for 1:
    pti_Weekday_AM1                           0.115657
    tt_ratio_Weekday_PrePM                    0.032492
    spd_Weekday_AM1                           0.027828
    tt_ratio_Weekday_AM2                      0.017655
    pti_Weekday_AM2                           0.016385


Waterfall saved: ../final_data/figures/shap_waterfall_2.png
  Top 5 SHAP features for 2:
    pti_Weekday_AM1                           0.224195
    tt_ratio_Weekday_PrePM                    0.036653
    tt_ratio_Weekday_AM2                      0.014358
    spd_Weekday_AM1                           0.010880
    pti_Weekday_AM2                           0.008553


Waterfall saved: ../final_data/figures/shap_waterfall_3.png
  Top 5 SHAP features for 3:
    pti_Weekday_AM1                           0.107714
    tt_ratio_Weekday_PrePM                    0.033042
    spd_Weekday_AM1                           0.027237
    tt_ratio_Weekday_AM2                      0.014763
    pti_Weekday_AM2                           0.009702


In [14]:
# ── 13. Full results table — all baselines × all targets (TEST set) ────────────

all_results = pd.DataFrame(results_lv + results_ha + results_sn + results_xgb)
print('All results shape:', all_results.shape)
print('Columns:', list(all_results.columns))

# Primary results table — test MAE per baseline × target
test_res = all_results[all_results['split'] == 'test'].copy()
pivot_mae = test_res.pivot_table(index='baseline', columns='target', values='MAE').round(6)
print(f'\n=== Test Set MAE — Baseline × Target ===')
print(pivot_mae.to_string())

pivot_rmse = test_res.pivot_table(index='baseline', columns='target', values='RMSE').round(6)
print(f'\n=== Test Set RMSE — Baseline × Target ===')
print(pivot_rmse.to_string())

pivot_mape = test_res.pivot_table(index='baseline', columns='target', values='MAPE').round(4)
print(f'\n=== Test Set MAPE (%) — Baseline × Target ===')
print(pivot_mape.to_string())

# Per-corridor results for primary target
print(f'\n=== Test MAE per corridor ({PRIMARY}) ===')
for cid in sorted(test['corr_id'].unique().tolist()):
    cid_str = str(cid)  # ensure Python str for f-string formatting
    test_c    = test[test['corr_id'] == cid]
    test_lv_c = test_lv[test_lv['corr_id'] == cid]
    test_ha_c = test_ha[test_ha['corr_id'] == cid]
    test_sn_c = test_sn[test_sn['corr_id'] == cid]

    # XGBoost predictions
    test_c_X = prepare_X(test_c)
    xgb_pred_c = model_xgb.predict(test_c_X)

    lv_pred_c = test_lv_c[f'{PRIMARY}_lv'].values
    ha_pred_c = test_ha_c[f'{PRIMARY}_ha'].values
    sn_pred_c = test_sn_c[f'{PRIMARY}_sn'].values
    true_c    = test_c[PRIMARY].values

    mae_lv  = mean_absolute_error(true_c, lv_pred_c)
    mae_ha  = float(np.nanmean(np.abs(true_c - ha_pred_c)))
    mae_sn  = float(np.nanmean(np.abs(true_c - sn_pred_c)))
    mae_xgb = mean_absolute_error(true_c, xgb_pred_c)

    print(f'  {cid_str:<25s}  LV={mae_lv:.6f}  HA={mae_ha:.6f}  SN={mae_sn:.6f}  XGB={mae_xgb:.6f}')

All results shape: (48, 8)
Columns: ['label', 'MAE', 'RMSE', 'MAPE', 'n', 'baseline', 'split', 'target']

=== Test Set MAE — Baseline × Target ===
target     pti_max  pti_mean  tt_ratio_Weekday_AM1  tt_ratio_Weekday_Midday  tt_ratio_Weekday_PM1  tt_ratio_Weekend_AM1
baseline                                                                                                               
HA        0.366250  0.191250              0.084583                 0.095833              0.076250              0.062083
LV        0.307083  0.101208              0.087917                 0.067083              0.055833              0.052500
SN        0.366250  0.191250              0.084583                 0.095833              0.076250              0.062083
XGB       0.172892  0.097632              0.046312                 0.068526              0.060745              0.041688

=== Test Set RMSE — Baseline × Target ===
target     pti_max  pti_mean  tt_ratio_Weekday_AM1  tt_ratio_Weekday_Midday  tt_ratio_Week

In [15]:
# ── 14. Visualisation — Predictions vs actuals (primary target, test set) ─────

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=False)
corridors_in_test = sorted(test['corr_id'].unique())

for i, cid in enumerate(corridors_in_test[:4]):
    ax = axes[i // 2][i % 2]
    
    test_c     = test[test['corr_id'] == cid].sort_values(['year','month'])
    test_lv_c  = test_lv[test_lv['corr_id'] == cid].sort_values(['year','month'])
    test_ha_c  = test_ha[test_ha['corr_id'] == cid].sort_values(['year','month'])
    test_sn_c  = test_sn[test_sn['corr_id'] == cid].sort_values(['year','month'])
    test_c_X   = prepare_X(test_c)
    
    x_axis = range(len(test_c))
    ax.plot(x_axis, test_c[PRIMARY].values,          'k-o', ms=4, lw=2,  label='Actual')
    ax.plot(x_axis, test_lv_c[f'{PRIMARY}_lv'].values,'r--', ms=3, lw=1,  label='LV')
    ax.plot(x_axis, test_ha_c[f'{PRIMARY}_ha'].values,'b--', ms=3, lw=1,  label='HA')
    ax.plot(x_axis, test_sn_c[f'{PRIMARY}_sn'].values,'g--', ms=3, lw=1,  label='SN')
    ax.plot(x_axis, model_xgb.predict(test_c_X),     'm-',  ms=3, lw=1,  label='XGB')
    
    months_labels = [f"{int(r['month']):02d}/{str(int(r['year']))[2:]}" for _, r in test_c.iterrows()]
    ax.set_xticks(list(x_axis))
    ax.set_xticklabels(months_labels, rotation=45, fontsize=7)
    ax.set_title(cid, fontsize=9)
    ax.set_ylabel(PRIMARY, fontsize=8)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(alpha=0.3)

plt.suptitle(f'Baselines vs Actual — {PRIMARY} (Test set 2024)', fontsize=12, y=1.01)
plt.tight_layout()
pred_path = FIG_DIR / 'baselines_predictions_test.png'
plt.savefig(pred_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Prediction plot saved: {pred_path}')

Prediction plot saved: ../final_data/figures/baselines_predictions_test.png


In [16]:
# ── 15. Save results to CSV ────────────────────────────────────────────────────

results_path = TABLE_DIR / 'baseline_results.csv'
all_results.to_csv(results_path, index=False)
print(f'Results saved: {results_path}  ({len(all_results)} rows)')

# Save SHAP importances
shap_df = shap_abs_mean.sort_values(ascending=False).reset_index()
shap_df.columns = ['feature', 'shap_mean_abs']
shap_path = TABLE_DIR / 'shap_feature_importance.csv'
shap_df.to_csv(shap_path, index=False)
print(f'SHAP importances saved: {shap_path}  ({len(shap_df)} features)')

print('\n=== SHAP Feature Importance (all features) ===')
print(shap_df.to_string())

Results saved: ../final_data/tables/baseline_results.csv  (48 rows)
SHAP importances saved: ../final_data/tables/shap_feature_importance.csv  (97 features)

=== SHAP Feature Importance (all features) ===
                       feature  shap_mean_abs
0              pti_Weekday_AM1       0.148170
1       tt_ratio_Weekday_PrePM       0.027127
2              spd_Weekday_AM1       0.023955
3         tt_ratio_Weekday_AM2       0.015368
4              pti_Weekday_AM2       0.014095
5            pti_Weekday_PrePM       0.003876
6       tt_ratio_Weekend_Night       0.003863
7       tt_ratio_Weekday_Early       0.003567
8                soc_phuket_l2       0.003107
9          pti_Weekend_Evening       0.002015
10           spd_Weekday_Early       0.001988
11        tt_ratio_Weekend_AM2       0.001979
12             spd_Weekday_AM2       0.001614
13             pti_Weekday_PM2       0.001551
14     tt_ratio_Weekend_Midday       0.001417
15             pti_Weekend_AM2       0.001267
16  tt_ratio_W

In [17]:
# ── 16. One-step-ahead re-evaluation — fair n=20 comparison with LLM ─────────
#
# The LLM predicts TTR(t+1) from context(t).
# To compare fairly, we re-train XGBoost in the same one-step-ahead mode:
#   X = features of month t  →  y = TTR of month t+1
#
# LV, HA, SN are also evaluated in one-step-ahead mode (predict t+1).
# All models are evaluated on the same 20 targets: Aug–Dec 2024
# (source months: Jul–Nov 2024).
#
# NOTE: this is a SEPARATE model from the same-period XGBoost above.
# The same-period XGBoost (MAE 0.049, n=24) remains the reference for its own task.

import itertools
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Build full data index for target lookup ────────────────────────────────────
all_ml = pd.concat([train, val, test], ignore_index=True)
all_idx = all_ml.set_index(['corr_id', 'year', 'month'])

def next_ym(yr, mo):
    return (yr + 1, 1) if mo == 12 else (yr, mo + 1)

def build_osa_pairs(df, target_col):
    """Return (X_df, y_array) where y = TTR of next month for each row in df."""
    keep_rows, y_vals = [], []
    for _, row in df.iterrows():
        nyr, nmo = next_ym(int(row['year']), int(row['month']))
        cid = row['corr_id']
        try:
            y_next = all_idx.loc[(cid, nyr, nmo), target_col]
            keep_rows.append(row)
            y_vals.append(y_next)
        except KeyError:
            pass   # last month of a split — no next target available
    return pd.DataFrame(keep_rows), np.array(y_vals)

# ── One-step-ahead splits ─────────────────────────────────────────────────────
# Train OSA: Apr–Nov 2023 → predict May–Dec 2023
train_osa_df, y_train_osa = build_osa_pairs(train, PRIMARY)
# Val OSA:   Jan–May 2024 → predict Feb–Jun 2024
val_osa_df,   y_val_osa   = build_osa_pairs(val,   PRIMARY)
# Test OSA:  Jul–Nov 2024 → predict Aug–Dec 2024  (n=20)
test_osa_df,  y_test_osa  = build_osa_pairs(test,  PRIMARY)

X_train_osa = prepare_X(train_osa_df)
X_val_osa   = prepare_X(val_osa_df)
X_test_osa  = prepare_X(test_osa_df)

print(f'OSA train: {X_train_osa.shape}  val: {X_val_osa.shape}  test: {X_test_osa.shape}')
print(f'n=20 check — test_osa rows: {len(y_test_osa)}')

# ── Optuna hyperparameter search for OSA XGBoost ──────────────────────────────
def xgb_osa_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 50, 300),
        'max_depth':        trial.suggest_int('max_depth', 2, 6),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'verbosity': 0, 'random_state': 42
    }
    m = xgb.XGBRegressor(**params)
    m.fit(X_train_osa, y_train_osa, eval_set=[(X_val_osa, y_val_osa)], verbose=False)
    return mean_absolute_error(y_val_osa, m.predict(X_val_osa))

study_osa = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_osa.optimize(xgb_osa_objective, n_trials=50, show_progress_bar=False)

best_osa = {**study_osa.best_params, 'verbosity': 0, 'random_state': 42}
model_xgb_osa = xgb.XGBRegressor(**best_osa)
model_xgb_osa.fit(X_train_osa, y_train_osa)

xgb_osa_val_mae  = mean_absolute_error(y_val_osa,  model_xgb_osa.predict(X_val_osa))
xgb_osa_test_mae = mean_absolute_error(y_test_osa, model_xgb_osa.predict(X_test_osa))
print(f'XGB-OSA  val MAE: {xgb_osa_val_mae:.4f}  |  test MAE: {xgb_osa_test_mae:.4f}')

# ── LV / HA / SN on n=20 (one-step-ahead, same 20 targets) ───────────────────
CORR_IDS   = sorted(test['corr_id'].unique())
master_idx = master.set_index(['corr_id', 'year', 'month'])

rows_n20 = []
for _, src_row in test_osa_df.iterrows():
    cid   = src_row['corr_id']
    src_m = int(src_row['month'])
    src_y = int(src_row['year'])
    tgt_y, tgt_m = next_ym(src_y, src_m)

    try:
        true_val = master_idx.loc[(cid, tgt_y, tgt_m), PRIMARY]
        src_val  = master_idx.loc[(cid, src_y, src_m), PRIMARY]
    except KeyError:
        continue

    # LV: last observed value
    pred_lv = src_val

    # HA: mean of target month across training years
    ha_mask = (train['corr_id'] == cid) & (train['month'] == tgt_m)
    pred_ha = train.loc[ha_mask, PRIMARY].mean() if ha_mask.any() else src_val

    # SN: same corridor + month, previous year
    try:
        pred_sn = master_idx.loc[(cid, tgt_y - 1, tgt_m), PRIMARY]
    except KeyError:
        pred_sn = pred_ha

    # XGB-OSA: one-step-ahead model
    pred_xgb = model_xgb_osa.predict(prepare_X(src_row.to_frame().T))[0]

    rows_n20.append({
        'corr_id': cid, 'source_month': src_m, 'target_month': tgt_m,
        'true': true_val,
        'pred_lv': pred_lv, 'pred_ha': pred_ha,
        'pred_sn': pred_sn, 'pred_xgb': pred_xgb
    })

df_n20 = pd.DataFrame(rows_n20)
print(f'\nn=20 evaluation rows: {len(df_n20)}')

# ── Compute and save metrics ───────────────────────────────────────────────────
results_n20 = []
print('\n=== n=20 one-step-ahead results (PRIMARY target) ===')
for baseline, col in [('LV','pred_lv'),('HA','pred_ha'),('SN','pred_sn'),('XGB','pred_xgb')]:
    mae  = (df_n20['true'] - df_n20[col]).abs().mean()
    rmse = ((df_n20['true'] - df_n20[col])**2).mean()**0.5
    mape = mape_safe(df_n20['true'].values, df_n20[col].values)
    results_n20.append({'model': baseline, 'MAE': round(mae, 3),
                        'RMSE': round(rmse, 3), 'MAPE': round(mape, 2), 'n': len(df_n20)})
    print(f'{baseline:5s}  MAE={mae:.3f}  RMSE={rmse:.3f}  MAPE={mape:.2f}%')

df_results_n20 = pd.DataFrame(results_n20)
n20_path = TABLE_DIR / 'baseline_results_n20.csv'
df_results_n20.to_csv(n20_path, index=False)
print(f'\nSaved: {n20_path}')

detail_n20_path = TABLE_DIR / 'baseline_details_n20.csv'
df_n20.to_csv(detail_n20_path, index=False)
print(f'Saved: {detail_n20_path}')

OSA train: (36, 97)  val: (24, 97)  test: (20, 97)
n=20 check — test_osa rows: 20


XGB-OSA  val MAE: 0.0896  |  test MAE: 0.0997

n=20 evaluation rows: 20

=== n=20 one-step-ahead results (PRIMARY target) ===
LV     MAE=0.145  RMSE=0.189  MAPE=10.19%
HA     MAE=0.088  RMSE=0.119  MAPE=5.92%
SN     MAE=0.088  RMSE=0.119  MAPE=5.92%
XGB    MAE=0.100  RMSE=0.129  MAPE=6.83%

Saved: ../final_data/tables/baseline_results_n20.csv
Saved: ../final_data/tables/baseline_details_n20.csv


In [18]:
# ── 16. Final Summary ──────────────────────────────────────────────────────────

print('=' * 70)
print('PHASE 3 — BASELINES SUMMARY')
print('=' * 70)

print('\n1. DATA OVERVIEW')
print(f'   Train: {train.shape}  |  Val: {val.shape}  |  Test: {test.shape}')
print(f'   Primary target: {PRIMARY}')
print(f'   Features used (XGB): {len(FEATURE_COLS)}')

print('\n2. TRAFFIC DATA — TEMPORAL VARIATION CHECK')
for t in TARGETS:
    std_by_corr = train.groupby('corr_id')[t].std().round(4)
    mean_by_corr = train.groupby('corr_id')[t].mean().round(4)
    print(f'   {t}: std={std_by_corr.values.tolist()} mean={mean_by_corr.values.tolist()}')
print('   → Non-zero std confirms traffic features vary month-to-month (dynamic TomTom data).')

print('\n3. TEST SET MAE SUMMARY (primary target)')
test_primary = test_res[test_res['target'] == PRIMARY][['baseline','MAE']].set_index('baseline')
print(test_primary.round(8).to_string())

print('\n4. SHAP INSIGHTS — TOP DRIVERS FOR CONGESTION')
top5_shap = shap_abs_mean.sort_values(ascending=False).head(5)
print('   Top 5 drivers (XGBoost learned these to distinguish corridors):')
for feat, imp in top5_shap.items():
    print(f'   → {feat}: {imp:.6f}')

print('\n5. IMPLICATIONS FOR THESIS')
print('   a) MAE values reflect real month-to-month variation (dynamic TomTom 2023-2024).')
print('   b) SHAP dominated by traffic-internal features (pti/tt_ratio/spd same timeset).')
print('      Exogenous (flights, weather, social) near-zero SHAP → XGB uses autocorrelation.')
print('      Interpretation: traffic is self-predictive at monthly granularity.')
print('   c) XGBoost baseline sets the performance floor for the LLM framework.')

print('\n6. FILES GENERATED')
generated = [
    results_path,
    shap_path,
    FIG_DIR / 'shap_importance_bar.png',
    FIG_DIR / 'shap_beeswarm.png',
    FIG_DIR / 'baselines_predictions_test.png',
]
for f in generated:
    exists = '✓' if pathlib.Path(f).exists() else '✗'
    print(f'   [{exists}] {f}')

print('\n' + '=' * 70)
print('Phase 3 complete — ready for Phase 4 (LLM Framework)')
print('=' * 70)

PHASE 3 — BASELINES SUMMARY

1. DATA OVERVIEW
   Train: (36, 106)  |  Val: (24, 106)  |  Test: (24, 106)
   Primary target: tt_ratio_Weekday_AM1
   Features used (XGB): 97

2. TRAFFIC DATA — TEMPORAL VARIATION CHECK
   tt_ratio_Weekday_AM1: std=[0.0816, 0.0931, 0.1631, 0.2244] mean=[1.2344, 1.1778, 1.5956, 1.5544]
   tt_ratio_Weekday_PM1: std=[0.0871, 0.0979, 0.1576, 0.1092] mean=[1.5078, 1.3311, 1.8367, 1.5422]
   tt_ratio_Weekday_Midday: std=[0.0773, 0.0559, 0.0723, 0.087] mean=[1.3033, 1.2456, 1.5933, 1.3611]
   tt_ratio_Weekend_AM1: std=[0.0936, 0.0408, 0.0574, 0.0304] mean=[1.1511, 0.9789, 1.2922, 1.1167]
   pti_mean: std=[0.1634, 0.1749, 0.2719, 0.0986] mean=[2.4716, 2.5566, 3.4337, 2.4359]
   pti_max: std=[0.2402, 0.46, 0.5185, 0.5191] mean=[3.2967, 3.4356, 4.8489, 3.6511]
   → Non-zero std confirms traffic features vary month-to-month (dynamic TomTom data).

3. TEST SET MAE SUMMARY (primary target)
               MAE
baseline          
LV        0.087917
HA        0.084583
SN  

## Key Findings — Phase 3

### Résultats baselines — Test Set MAE (`tt_ratio_Weekday_AM1`)
| Baseline | Val MAE | Test MAE | Test MAPE |
|----------|---------|----------|-----------|
| LV | 0.109 | 0.088 | 6.1% |
| HA | 0.090 (n=12*) | 0.085 | 5.6% |
| SN | 0.094 | 0.085 | 5.6% |
| **XGB** | **0.035** | **0.048** | **3.2%** |

*HA val n=12/24 : Jan–Mar 2024 absents du train (Apr–Dec 2023 seulement).

### Données
- **Train :** Apr–Dec 2023 → 9 mois × 4 corridors = **36 lignes** (Jan–Mar droppés : pas de lags L1/L2/L3)
- **Val :** Jan–Jun 2024 → 6 mois × 4 corridors = **24 lignes**
- **Test :** Jul–Dec 2024 → 6 mois × 4 corridors = **24 lignes**
- **Trafic :** TomTom MOVE — dynamique, 24 timesets réels, std > 0 sur toutes les cibles ✅
- **Features XGB :** trafic interne + lags + météo + vols + social + calendrier. Interpréter comme baseline forte, pas exogène-only.

### Interprétation
- **HA ≡ SN sur test** : confirmé (1 seule année train → lookup identique)
- **XGB bat tous les baselines** : MAE 0.048 vs 0.085–0.088 → ×1.8 amélioration
- **SHAP : features trafic-internes dominent** — `pti_Weekday_AM1` (0.123), `tt_ratio_Weekday_PrePM` (0.051), `spd_Weekday_AM1` (0.029)
- **Exogènes quasi-nuls** : `flt_total_pax_l2` rang 11 (0.003) → à granularité mensuelle, le trafic est auto-prédictif

### Implications thèse
1. **Baseline XGBoost** : MAE test 0.048 = plancher de performance pour le LLM framework (Phase 4)
2. **SHAP** : dominance trafic-interne → le LLM doit exploiter la structure temporelle des timesets
3. **Limitation** : 36 train rows → overfitting probable (train MAE 0.011 vs test 0.048)
4. **Extension** : plus d’historique TomTom (ex. 36 mois) → plus de train rows, MAE plus fiables, exogènes peut-être plus visibles
